[![在 Colab 中打开](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Arize-ai/phoenix/blob/main/tutorials/prompts/prompt-optimization.ipynb)


<center>
<p style="text-align:center">
<img alt="Phoenix 标志" src="https://raw.githubusercontent.com/Arize-ai/phoenix-assets/9e6101d95936f4bd4d390efc9ce646dc6937fb2d/images/socal/github-large-banner-phoenix.jpg" width="1000"/>
<br>
<br>
<a href="https://arize.com/docs/phoenix/">文档</a>
|
<a href="https://github.com/Arize-ai/phoenix">GitHub</a>
|
<a href="https://join.slack.com/t/arize-ai/shared_invite/zt-3r07iavnk-ammtATWSlF0pSrd1DsMW7g">社区</a>
</p>
</center>
<h1 align="center">比较提示词优化技术</h1>

本教程使用 Phoenix 比较不同提示词优化技术的表现。

首先在 Phoenix 中创建实验，保存各个优化后的提示词对应的结果；随后使用多种技术改善越狱分类任务的效果。每种技术都基于同一个初始提示词，最终通过 Phoenix 比较结果。

将使用以下技术：
- **少样本示例**：向提示词加入若干示例，帮助模型理解任务。
- **元提示**：根据先前的输入、输出和预期输出，让模型生成更好的提示词。
- **提示词梯度**：利用嵌入表示和提示词梯度优化提示词的各个部分。
- **DSPy 提示词调优**：使用自动提示词调优库 DSPy 优化提示词。

⚠️ 本教程需要 OpenAI API 密钥。

开始吧！


### 配置依赖与密钥

In [ ]:
!pip install -q "arize-phoenix>=8.0.0" datasets

接下来连接 Phoenix。下面的代码连接到 Phoenix Cloud 实例；你也可以选择[连接自托管 Phoenix 实例](https://arize.com/docs/phoenix/deployment)。

In [ ]:
import os
from getpass import getpass

os.environ["PHOENIX_COLLECTOR_ENDPOINT"] = "https://app.phoenix.arize.com"
if not os.environ.get("PHOENIX_CLIENT_HEADERS"):
    os.environ["PHOENIX_CLIENT_HEADERS"] = "api_key=" + getpass("请输入 Phoenix API 密钥：")

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("请输入 OpenAI API 密钥：")

### 将数据集加载到 Phoenix

由于要运行一系列实验，需要准备一份每次都能复用的测试用例数据集，用来评估各种提示词优化技术的效果。

In [ ]:
from datasets import load_dataset

ds = load_dataset("jackhhao/jailbreak-classification")["train"]
ds = ds.to_pandas().sample(50)
ds.head()

In [ ]:
import uuid

from phoenix.client import Client
from phoenix.client import Client as PhoenixClient

unique_id = uuid.uuid4()

# 将数据集上传到 Phoenix
dataset = Client().datasets.create_dataset(
    dataframe=ds,
    input_keys=["prompt"],
    output_keys=["type"],
    name=f"jailbreak-classification-{unique_id}",
)

接下来定义初始提示词模板，并保存到 Phoenix，以便跟踪、管理版本和跨实验复用。

In [ ]:
from openai import OpenAI
from openai.types.chat.completion_create_params import CompletionCreateParamsBase
from phoenix.client.types import PromptVersion

params = CompletionCreateParamsBase(
    model="gpt-3.5-turbo",
    temperature=0,
    messages=[
        {
            "role": "system",
            "content": "你是评估器，负责判断给定提示词是否存在越狱风险。只能输出英文标签 'benign' 或 'jailbreak'，不要输出其他词。",
        },
        {"role": "user", "content": "{{prompt}}"},
    ],
)

prompt_identifier = "jailbreak-classification"

prompt = PhoenixClient().prompts.create(
    name=prompt_identifier,
    prompt_description="判断给定提示词是否存在越狱风险的提示词。",
    version=PromptVersion.from_openai(params),
)

现在应能在 Phoenix 中看到该提示词：

![Phoenix 中的提示词](https://storage.googleapis.com/arize-phoenix-assets/assets/images/prompts-screenshot-prompt-optimization-2.png)

接下来为实验准备任务和评估器。任务是对数据集中的每个样本执行的函数，也是各轮实验间需要修改的代码部分。起初，任务只是使用基础提示词调用 GPT-3.5 Turbo。

还需要评估器来衡量任务表现。每个样本的任务执行完成后，评估器会对结果进行检查。由于数据集包含真实标签，这里可用简单函数判断任务输出是否符合预期。

In [ ]:
def test_prompt(input):
    client = OpenAI()
    resp = client.chat.completions.create(**prompt.format(variables={"prompt": input["prompt"]}))
    return resp.choices[0].message.content.strip()


def evaluate_response(output, expected):
    return output.lower() == expected["type"].lower()

你也可以为代码接入埋点，把所有模型调用发送到 Phoenix。实验本身不依赖埋点，但接入后可在 Phoenix 中查看每次实验任务的追踪。无论是否埋点，实验总分和评估器运行结果都会被记录。

In [ ]:
from openinference.instrumentation.openai import OpenAIInstrumentor
from phoenix.otel import register

tracer_provider = register(project_name="prompt-optimization")
OpenAIInstrumentor().instrument(tracer_provider=tracer_provider)

现在运行初始实验。后续会基于这个初始提示词进行优化。

In [ ]:
import nest_asyncio
from phoenix.client.experiments import run_experiment

nest_asyncio.apply()

initial_experiment = run_experiment(
    dataset,
    task=test_prompt,
    evaluators=[evaluate_response],
    experiment_description="初始基础提示词",
    experiment_name="initial-prompt",
    experiment_metadata={"prompt": "prompt_id=" + prompt.id},
)

现在应能在 Phoenix 中看到初始实验结果：

![首次实验结果](https://storage.googleapis.com/arize-phoenix-assets/assets/images/prompts-nb-experiment.png)

# 提示词优化技术一：少样本示例

一种常见方法是在提示词中加入少样本示例，引导模型行为。

这里将加入示例来改善效果。上一步上传的数据集中恰好包含可用于此目的的测试集。

In [ ]:
from datasets import load_dataset

ds_test = load_dataset("jackhhao/jailbreak-classification")[
    "test"
]  # 本次加载测试集，而非训练集
few_shot_examples = ds_test.to_pandas().sample(10)

定义包含少样本示例的新提示词。Phoenix 自动管理提示词版本，因此以相同名称保存时会创建一个可供使用的新版本。

In [ ]:
few_shot_template = """
你是评估器，负责判断给定提示词是否存在越狱风险。只能输出英文标签 "benign" 或 "jailbreak"，不要输出其他词。

下面是提示词与回答的示例：

{examples}
"""

params = CompletionCreateParamsBase(
    model="gpt-3.5-turbo",
    temperature=0,
    messages=[
        {"role": "system", "content": few_shot_template.format(examples=few_shot_examples)},
        {"role": "user", "content": "{{prompt}}"},
    ],
)

few_shot_prompt = PhoenixClient().prompts.create(
    name=prompt_identifier,
    prompt_description="少样本提示词",
    version=PromptVersion.from_openai(params),
)

现在可以在 Phoenix 中看到新的提示词版本：

![Phoenix 中的少样本提示词](https://storage.googleapis.com/arize-phoenix-assets/assets/images/prompt-versioning-nb.png)

使用新提示词定义一个新任务：

In [ ]:
def test_prompt(input):
    client = OpenAI()
    prompt_vars = {"prompt": input["prompt"]}
    resp = client.chat.completions.create(**few_shot_prompt.format(variables=prompt_vars))
    return resp.choices[0].message.content.strip()

现在可以使用新提示词再次运行实验。测试数据集和评估器与上一次实验保持一致。

In [ ]:
few_shot_experiment = run_experiment(
    dataset,
    task=test_prompt,
    evaluators=[evaluate_response],
    experiment_description="提示词优化技术一：少样本示例",
    experiment_name="few-shot-examples",
    experiment_metadata={"prompt": "prompt_id=" + few_shot_prompt.id},
)

# 提示词优化技术二：元提示

元提示根据先前的输入、输出和预期输出，要求模型生成更好的提示词。

第一轮实验记录了上述所有信息，因此很适合作为这种方法的起点。

In [ ]:
# 将第一轮实验结果读取为 DataFrame
ground_truth_df = initial_experiment.as_dataframe()

# 抽取 10 个样本用作元提示示例
ground_truth_df = ground_truth_df[:10]

# 新建一列，将示例整合为一个字符串
ground_truth_df["example"] = ground_truth_df.apply(
    lambda row: (
        f"输入：{row['input']}\n输出：{row['output']}\n预期输出：{row['expected']}"
    ),
    axis=1,
)
ground_truth_df.head()

接下来构建一条用于生成新提示词的提示词。

In [ ]:
meta_prompt = """
你是一名提示词工程专家。下面提供一条提示词和一组示例。

你的任务是生成一条能改善模型表现的新提示词。

示例如下：

{examples}

原始提示词如下：

{prompt}

新提示词如下：
"""

original_base_prompt = (
    prompt.format(variables={"prompt": "example prompt"}).get("messages")[0].get("content")
)

client = OpenAI()
response = client.chat.completions.create(
    model="gpt-3.5-turbo",
    messages=[
        {
            "role": "user",
            "content": meta_prompt.format(
                prompt=original_base_prompt, examples=ground_truth_df["example"].to_string()
            ),
        }
    ],
)
new_prompt = response.choices[0].message.content.strip()

In [ ]:
new_prompt

将生成的提示词保存到 Phoenix：

In [ ]:
if r"\{examples\}" in new_prompt:
    new_prompt = new_prompt.format(examples=few_shot_examples)

params = CompletionCreateParamsBase(
    model="gpt-3.5-turbo",
    temperature=0,
    messages=[
        {"role": "system", "content": new_prompt},
        {"role": "user", "content": "{{prompt}}"},
    ],
)

meta_prompt_result = PhoenixClient().prompts.create(
    name=prompt_identifier,
    prompt_description="元提示结果",
    version=PromptVersion.from_openai(params),
)

### 使用新提示词运行同一实验
使用新提示词重新定义任务。

In [ ]:
def test_prompt(input):
    client = OpenAI()
    resp = client.chat.completions.create(
        **meta_prompt_result.format(variables={"prompt": input["prompt"]})
    )
    return resp.choices[0].message.content.strip()

In [ ]:
meta_prompting_experiment = run_experiment(
    dataset,
    task=test_prompt,
    evaluators=[evaluate_response],
    experiment_description="提示词优化技术二：元提示",
    experiment_name="meta-prompting",
    experiment_metadata={"prompt": "prompt_id=" + meta_prompt_result.id},
)

# 提示词优化技术三：提示词梯度优化

提示词梯度优化利用嵌入表示和提示词梯度调整提示词的各个部分，步骤如下：
1. 将提示词转换为嵌入向量。
2. 比较成功与失败提示词的输出，确定梯度方向。
3. 沿该方向调整提示词。

这里将定义获取提示词嵌入向量的函数，并用它计算成功与失败提示词之间的梯度方向。

In [ ]:
import numpy as np


# 先定义获取提示词嵌入向量的函数
def get_embedding(text):
    client = OpenAI()
    response = client.embeddings.create(model="text-embedding-ada-002", input=text)
    return response.data[0].embedding


# 计算成功与失败提示词之间梯度方向的函数
def calculate_prompt_gradient(successful_prompts, failed_prompts):
    # 获取成功与失败提示词的嵌入向量
    successful_embeddings = [get_embedding(p) for p in successful_prompts]
    failed_embeddings = [get_embedding(p) for p in failed_prompts]

    # 计算平均嵌入向量
    avg_successful = np.mean(successful_embeddings, axis=0)
    avg_failed = np.mean(failed_embeddings, axis=0)

    # 计算梯度方向
    gradient = avg_successful - avg_failed
    return gradient / np.linalg.norm(gradient)


# 从数据集中获取成功和失败样本
successful_examples = (
    ground_truth_df[ground_truth_df["output"] == ground_truth_df["expected"].get("type")]["input"]
    .apply(lambda x: x["prompt"])
    .tolist()
)
failed_examples = (
    ground_truth_df[ground_truth_df["output"] != ground_truth_df["expected"].get("type")]["input"]
    .apply(lambda x: x["prompt"])
    .tolist()
)

# 计算梯度方向
gradient = calculate_prompt_gradient(successful_examples[:5], failed_examples[:5])


# 利用梯度优化提示词的函数
def optimize_prompt(base_prompt, gradient, step_size=0.1):
    # 获取初始嵌入向量
    base_embedding = get_embedding(base_prompt)

    # 沿梯度方向移动
    optimized_embedding = base_embedding + step_size * gradient

    # 使用 GPT 将优化后的嵌入向量转换回文本
    client = OpenAI()
    response = client.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=[
            {
                "role": "system",
                "content": "你的任务是优化提示词。根据原始提示词及其嵌入向量，生成保留核心含义、同时朝优化后的嵌入方向调整的新版本。",
            },
            {
                "role": "user",
                "content": f"原始提示词：{base_prompt}\n优化后的嵌入方向：{optimized_embedding[:10]}...\n请生成一个沿此嵌入方向改进的版本。",
            },
        ],
    )
    return response.choices[0].message.content.strip()


# 测试基于梯度的优化
gradient_prompt = optimize_prompt(original_base_prompt, gradient)

In [ ]:
gradient_prompt

In [ ]:
if r"\{examples\}" in gradient_prompt:
    gradient_prompt = gradient_prompt.format(examples=few_shot_examples)

params = CompletionCreateParamsBase(
    model="gpt-3.5-turbo",
    temperature=0,
    messages=[
        {
            "role": "system",
            "content": gradient_prompt,
        },  # 如果元提示包含少样本示例，请在这里一并加入
        {"role": "user", "content": "{{prompt}}"},
    ],
)

gradient_prompt = PhoenixClient().prompts.create(
    name=prompt_identifier,
    prompt_description="梯度提示结果",
    version=PromptVersion.from_openai(params),
)

### 使用梯度优化后的提示词运行实验
使用新提示词重新定义任务。

In [ ]:
def test_gradient_prompt(input):
    client = OpenAI()
    resp = client.chat.completions.create(
        **gradient_prompt.format(variables={"prompt": input["prompt"]})
    )
    return resp.choices[0].message.content.strip()

In [ ]:
gradient_experiment = run_experiment(
    dataset,
    task=test_gradient_prompt,
    evaluators=[evaluate_response],
    experiment_description="提示词优化技术三：提示词梯度",
    experiment_name="gradient-optimization",
    experiment_metadata={"prompt": "prompt_id=" + gradient_prompt.id},
)

# 提示词优化技术四：使用 DSPy 调优

最后，还可以使用 DSPy 这类优化库改进提示词。[DSPy](https://github.com/stanfordnlp/dspy) 支持前面介绍的技术，以及更多方法。

In [ ]:
!pip install -q dspy openinference-instrumentation-dspy

DSPy 会通过一系列调用优化提示词。观察这些调用有助于理解优化过程：可以使用 OpenInference SDK 为 DSPy 接入埋点，将所有调用发送到 Phoenix。这一步是可选的。

In [ ]:
from openinference.instrumentation.dspy import DSPyInstrumentor

DSPyInstrumentor().instrument(tracer_provider=tracer_provider)

现在配置 DSPy 的语言模型，并定义提示词分类任务。

In [ ]:
# 导入 DSPy 并配置语言模型
import dspy

# 配置 DSPy 使用 OpenAI
turbo = dspy.LM(model="gpt-3.5-turbo")
dspy.settings.configure(lm=turbo)


# 定义提示词分类任务
class PromptClassifier(dspy.Signature):
    """判断提示词属于 benign（正常）还是 jailbreak（越狱）。"""

    prompt = dspy.InputField()
    label = dspy.OutputField(desc="只能为 'benign' 或 'jailbreak'")


# 创建基础分类器
classifier = dspy.Predict(PromptClassifier)

分类器现在可以像普通 LLM 一样用于预测：它接收 `prompt` 输入，输出 `label` 预测。

In [ ]:
classifier(prompt=ds.iloc[0].prompt)

DSPy 的主要优势在于提示词优化。只要定义衡量任务成功的指标，并提供训练样本，就可以使用库中内置的多种优化器。

这里使用 `MIPROv2` 优化器，为任务寻找最佳提示词。

In [ ]:
def validate_classification(example, prediction, trace=None):
    return example["label"] == prediction["label"]


# 根据前面的样本准备训练数据
train_data = []
for _, row in ground_truth_df.iterrows():
    example = dspy.Example(
        prompt=row["input"]["prompt"], label=row["expected"]["type"]
    ).with_inputs("prompt")
    train_data.append(example)

tp = dspy.MIPROv2(metric=validate_classification, auto="light")
optimized_classifier = tp.compile(classifier, trainset=train_data)

此时提示词由 DSPy 管理，但仍可将生成的提示词内容保存到 Phoenix：

In [ ]:
params = CompletionCreateParamsBase(
    model="gpt-3.5-turbo",
    temperature=0,
    messages=[
        {
            "role": "system",
            "content": optimized_classifier.signature.instructions,
        },  # 如果元提示包含少样本示例，请在这里一并加入
        {"role": "user", "content": "{{prompt}}"},
    ],
)

dspy_prompt = PhoenixClient().prompts.create(
    name=prompt_identifier,
    prompt_description="DSPy 提示结果",
    version=PromptVersion.from_openai(params),
)

### 使用 DSPy 优化的分类器运行实验
使用新提示词重新定义任务。

In [ ]:
# 使用优化后的分类器创建评测函数
def test_dspy_prompt(input):
    result = optimized_classifier(prompt=input["prompt"])
    return result.label

In [ ]:
# 使用 DSPy 优化后的分类器运行实验
dspy_experiment = run_experiment(
    dataset,
    task=test_dspy_prompt,
    evaluators=[evaluate_response],
    experiment_description="提示词优化技术四：DSPy 提示词调优",
    experiment_name="dspy-optimization",
    experiment_metadata={"prompt": "prompt_id=" + dspy_prompt.id},
)

# 提示词优化技术五：DSPy 配合 GPT-4o

上一个示例同时使用 GPT-3.5 Turbo 执行流程和优化提示词。你也可以为优化与执行分别选择不同模型。

一种实用方式是用更强的模型优化提示词，再用更便宜或更快的模型执行流程。

这里使用 GPT-4o 优化提示词，执行模型仍保留 GPT-3.5 Turbo。

In [ ]:
prompt_gen_lm = dspy.LM("gpt-4o")
tp = dspy.MIPROv2(
    metric=validate_classification, auto="light", prompt_model=prompt_gen_lm, task_model=turbo
)
optimized_classifier_using_gpt_4o = tp.compile(classifier, trainset=train_data)

### 使用 GPT-4o 配合 DSPy 优化的分类器运行实验
使用新提示词重新定义任务。

In [ ]:
# 使用优化后的分类器创建评测函数
def test_dspy_prompt(input):
    result = optimized_classifier_using_gpt_4o(prompt=input["prompt"])
    return result.label

In [ ]:
# 使用 DSPy 优化后的分类器运行实验
dspy_experiment_using_gpt_4o = run_experiment(
    dataset,
    task=test_dspy_prompt,
    evaluators=[evaluate_response],
    experiment_description="提示词优化技术五：DSPy 配合 GPT-4o 调优",
    experiment_name="dspy-optimization-gpt-4o",
    experiment_metadata={"prompt": "prompt_id=" + dspy_prompt.id},
)

# 完成！

你已使用多种提示词优化技术改善越狱分类任务的表现，并通过 Phoenix 比较结果。

现在应该有一组类似下图的实验：

![实验结果](https://storage.googleapis.com/arize-phoenix-assets/assets/images/prompt-optimization-experiment-screenshot.png)

接下来可以查看更多 [Phoenix 示例](https://arize.com/docs/phoenix/notebooks)。如果还没有为项目加星，也欢迎[在 GitHub 上支持我们](https://github.com/Arize-ai/phoenix)！⭐️